Intento de detección de matrículas basada en contornos.

In [ ]:
import cv2  
import math 

from ultralytics import YOLO

# Carga del modelo
model = YOLO('yolo11n.pt') #Contenedores

#Para un vídeo
filename = "C0142.mp4"

cap = cv2.VideoCapture(filename)

cv2.namedWindow('Deteccion con YOLO', cv2.WINDOW_NORMAL)
cv2.resizeWindow('Deteccion con YOLO', 1280, 720)

# funcion para detectar matrículas
def detect_plate(car_region):
    # paso la imagen a escala de grises
    gris = cv2.cvtColor(car_region, cv2.COLOR_BGR2GRAY)

    # suavizo para reducir el ruido
    gris = cv2.GaussianBlur(gris, (5, 5), 0)

    # aplico umbral adaptativo para destacar los bordes
    img_th1 = cv2.adaptiveThreshold(gris, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 21, 5) 

    # obtengo los contornos externos
    contornos, _ = cv2.findContours(img_th1, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # comprobamos los contornos
    candidatos_mat = []
    for contorno in contornos:
        # aproximo el contorno a un polígono
        perimeter = cv2.arcLength(contorno, True)
        aprox = cv2.approxPolyDP(contorno, 0.018 * perimeter, True)

        # obtengo el rectángulo
        x, y, w, h = cv2.boundingRect(aprox)

        # características para identificar la matrícula
        aspect_ratio = w / float(h)
        area = w * h
        car_area = car_region.shape[0] * car_region.shape[1]
        relative_area = area / car_area

        if (2.0 <= aspect_ratio <= 5.5 and 
            0.01 <= relative_area <= 0.15 and
            w > 40 and h > 10):

            # Puntuación basada en qué tan cerca está del ratio ideal
            ideal_ratio = 4.5
            ratio_score = 1 - abs(aspect_ratio - ideal_ratio) / ideal_ratio
            
            candidatos_mat.append({
                'contour': aprox,
                'bbox': (x, y, w, h),
                'score': ratio_score * relative_area,
                'aspect_ratio': aspect_ratio
            })

    # cogemos el mejor candidato
    if candidatos_mat:
        mejor_matricula = max(candidatos_mat, key=lambda x: x['score'])
        return mejor_matricula

while cap.isOpened():
    ret, frame = cap.read()

    # si no hay imagen salimos
    if not ret:
        break

    # se ejecuta el modelo en el frame y se añaden los recuadros
    results = model(frame, classes=[2, 3, 5, 7], conf=0.5)
    annotated_frame = results[0].plot()

    # busco las matrículas de cada vehículo
    for result in results[0].boxes.data:
        x1, y1, x2, y2, conf, cls = result
        x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
        
        # Extraer región del vehículo
        car_region = frame[y1:y2, x1:x2].copy()

        if car_region.size > 0:
            # busco la matricula
            plate = detect_plate(car_region)
            
            # si la encuentro, la dibujo
            if plate is not None:
                detections_count += 1
                px, py, pw, ph = plate['bbox']
                
                # Ajustar coordenadas al frame completo
                px += x1
                py += y1
                
                # Dibujar rectángulo de la matrícula en rojo
                cv2.rectangle(annotated_frame, (px, py), (px + pw, py + ph), (0, 0, 255), 2)
                cv2.putText(annotated_frame, 'PLATE', (px, py - 5),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
    
    frame = annotated_frame

    cv2.imshow('Deteccion con YOLO', annotated_frame)

    # se sale con ESC o Q/q
    key = cv2.waitKey(1)
    print(key)
    if key == 27 or key == 81 or key == 113:
        break

cap.release()
cv2.destroyAllWindows()

Modelo YOLO para matrículas

In [ ]:
"""
from ultralytics import YOLO

# Cargar modelo preentrenado
model = YOLO("yolo11n.pt")

# Entrenar
model.train(
    data="dataset.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    name="yolo_matriculas",
    device=0
)

# Predicción en validación
results = model.predict(source='C:/Users/juanf/Desktop/Large-License-Plate-Detection-Dataset/images/val', save=True)
"""

Instalación de dependencias

In [ ]:

from ultralytics import YOLO # Importamos la librería ultralytics para usar el modelo YOLO y entrenar nuestro detector de matrículas
import cv2 # Importamos OpenCV para procesamiento de imágenes
from collections import defaultdict # Importamos defaultdict para manejar diccionarios con listas
import csv # Importamos csv para manejar archivos CSV
from PIL import Image # Importamos PIL para manejar imágenes
import torch # Importamos PyTorch para manejo de tensores y modelos
from transformers import AutoProcessor, AutoModelForVision2Seq # Importamos transformers para modelos de visión a secuencia
import numpy as np  # Importamos numpy para manejo de arreglos numéricos

 CONFIGURACIÓN DEL MODELO smolVLM para OCR

In [ ]:

#establecer dispositivo para el modelo
device = "cuda" if torch.cuda.is_available() else "cpu"

#cargar modelo SmolVLM-Instruct que es adecuado para tareas de visión a secuencia
model_name = "HuggingFaceTB/SmolVLM-Instruct"

#cargar el procesador asociado al modelo
processor = AutoProcessor.from_pretrained(model_name)

#cargar el modelo con el tipo de dato adecuado según el dispositivo
model = AutoModelForVision2Seq.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)


FUNCIÓN PARA EXTRAER TEXTO DE MATRÍCULA CON smolVLM

In [ ]:
def extraer_texto_matricula(frame, x1, y1, x2, y2):

    try:
        # Extraemos la región de interés (ROI) de la matrícula con un pequeño padding
        padding = 5
        roi = frame[max(0, y1-padding):min(frame.shape[0], y2+padding), 
                   max(0, x1-padding):min(frame.shape[1], x2+padding)]
        
        # Verificamos si la ROI está vacía
        if roi.size == 0:
            return ""

        # Convertimos de BGR (OpenCV) a RGB (PIL)
        roi_rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
        pil_image = Image.fromarray(roi_rgb)
        
        # Prompt específico para lectura de matrículas
        prompt = "Read the license plate number in this image. Only output the alphanumeric characters you see, without spaces or additional text."
        
        # Preparamos la plantilla de entrada para el modelo (imagen + prompt)
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": prompt}
                ]
            }
        ]

        # Aplicamos la plantilla
        text = processor.apply_chat_template(messages, add_generation_prompt=True)

        # Cargamos la imagen y el texto al procesador
        inputs = processor(text=[text], images=[pil_image], return_tensors="pt")

        # Movemos los tensores al dispositivo adecuado
        inputs = inputs.to(device)
        
        # Generamos la predicción sin calcular gradientes(no es entrenamiento)
        with torch.no_grad():
            generated_ids = model.generate(
                **inputs, # Desempaquetamos los inputs
                max_new_tokens=15, # Máximo número de tokens a generar
                do_sample=False 
            )

        # Decodificamos resultado
        generated_texts = processor.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )

        # Extraemos solo el texto de la respuesta
        texto = generated_texts[0].split("Assistant:")[-1].strip()

        # Limpiamos el texto: solo alfanuméricos
        texto_limpio = ''.join(c for c in texto if c.isalnum()).upper()
        
        return texto_limpio
    
    except Exception as e:
        print(f"Error al procesar matrícula: {e}")
        return ""

FUNCIÓN PARA ASOCIAR MATRÍCULAS CON VEHÍCULOS

In [ ]:
def asociar_matricula_con_vehiculo(plate_box, vehicle_boxes):

    # Obtenemos el centro de la matrícula
    px1, py1, px2, py2 = plate_box
    plate_center_x = (px1 + px2) / 2
    plate_center_y = (py1 + py2) / 2
    
    # Inciamos variables auxiliares para encontrar el mejor vehículo
    mejor_vehiculo = None
    mejor_distancia = float('inf')
    
    # Recorremos las cajas de vehículos
    for idx, vbox in enumerate(vehicle_boxes):
        vx1, vy1, vx2, vy2 = vbox
        
        # Verificamos si la matrícula está dentro del vehículo( y si es así, la asociamos directamente)
        if vx1 <= plate_center_x <= vx2 and vy1 <= plate_center_y <= vy2:
            return idx
        
        # Calculamos distancia al centro del vehículo
        vcenter_x = (vx1 + vx2) / 2
        vcenter_y = (vy1 + vy2) / 2
        distancia = np.sqrt((plate_center_x - vcenter_x)**2 + (plate_center_y - vcenter_y)**2)
        
        # Actualizamos el mejor vehículo si la distancia es menor
        if distancia < mejor_distancia:
            mejor_distancia = distancia
            mejor_vehiculo = idx
    
    # Solo vamos a asociar si la distancia es razonable (menos de 200 píxeles)
    if mejor_distancia < 200:
        return mejor_vehiculo
    return None

CARGA DE MODELOS YOLO

In [ ]:
# Cargamos el modelo general de detección de objetos YOLOv11
general = YOLO("yolo11n.pt")

# Cargamos el modelo específico entrenado para detección de matrículas
matriculas = YOLO("runs/detect/yolo_matriculas/weights/best.pt")

CONFIGURACIÓN DE PARÁMETROS

In [ ]:

video_path = "C0142.MP4"
output_path = "detecciones_combinadas_final.mp4"
csv_path = "detecciones_tracking_matriculas.csv"
tracker = "bytetrack.yaml"

conf_general = 0.5
conf_plate = 0.3
classes_general = [0, 2, 3, 5, 7]  # person, car, motorcycle, bus, truck

# OPTIMIZACIÓN: Procesar OCR solo cada N frames
OCR_CADA_N_FRAMES = 5  # Ajusta este valor (5 = cada 5 frames, 10 = más rápido pero menos preciso)

PROCESAMIENTO DEL VIDEO

In [ ]:
# Declaramos un diccionario para almacenar la mejor predicción de cada matrícula por vehículo
# key: track_id del vehículo, value: { 'frame', 'bbox', 'texto', 'conf' }
mejores_matriculas_por_vehiculo = {}

# Diccionario para almacenar los IDs únicos por clase detectada
ids_por_clase = defaultdict(set)
 
# Abrimos el archivo CSV para escritura
with open(csv_path, 'w', newline='', encoding='utf-8') as f_csv:
    csv_writer = csv.writer(f_csv)
    csv_writer.writerow([
        'frame', 'tipo', 'conf_obj', 'track_id', 'x1', 'y1', 'x2', 'y2',
        'mat_detectada', 'conf_mat', 'mat_x1', 'mat_y1', 'mat_x2', 'mat_y2', 'texto_matricula'
    ])
    
    # Lanzamos el tracking con el modelo general, modo stream ya que es más eficiente para videos largos
    results_stream = general.track(
        source=video_path,
        tracker=tracker,
        classes=classes_general,
        conf=conf_general,
        persist=True,
        stream=True
    )
    
    # Definimos variables auxiliares
    h, w, fps = None, None, 30
    writer = None
    objetos_por_frame = []  # Lista para guardar los objetos de cada frame
    
    print("PROCESANDO VIDEO Y GENERANDO SALIDAS")
    
    # Recorremos los resultados frame a frame del resultados del tracking
    for frame_num, r in enumerate(results_stream):
        
        # Copiamos el frame original y obtenemos el frame con tracking dibujado
        frame = r.orig_img.copy()
        tracked_frame = r.plot()

        # Inicializamos writer la primera vez para crear el video de salida
        if writer is None:
            h, w = frame.shape[:2]
            writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
        
        # Declaramos lista para almacenar los objetos trackeados en el frame
        objetos_trackeados = []

        # Si el frame tiene cajas detectadas, las recorremos
        if hasattr(r, "boxes") and r.boxes is not None:
            for box in r.boxes:

                # Nos saltamos cajas sin ID asignado
                if box.id is None:
                    continue

                # Obtenemos los datos de la caja
                cls = int(box.cls)
                track_id = int(box.id)
                conf = float(box.conf)
                x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

                # Actualizamos el conjunto de IDs por clase
                ids_por_clase[cls].add(track_id)

                # Almacenamos el objeto trackeado
                objetos_trackeados.append({
                    'cls': cls,
                    'track_id': track_id,
                    'conf': conf,
                    'bbox': (x1, y1, x2, y2),
                    'tipo': general.names[cls]
                })
        
        # Declaramos lista para almacenar las matrículas detectadas en el frame
        matriculas_detectadas = []

        # Procesamos OCR solo cada N frames
        if frame_num % OCR_CADA_N_FRAMES == 0:

            # Realizamos la detección de matrículas en el frame actual
            res_plate = matriculas(frame, conf=conf_plate, verbose=False)[0]

            # Si hay resultados de detección de matrículas, los recorremos
            if hasattr(res_plate, "boxes") and res_plate.boxes is not None:
                for pbox in res_plate.boxes:

                    # Obtenemos las coordenadas, confianza y texto de la matrícula
                    x1, y1, x2, y2 = map(int, pbox.xyxy[0].tolist())
                    conf_plate_det = float(pbox.conf)
                    texto_matricula = extraer_texto_matricula(frame, x1, y1, x2, y2)

                    # Almacenamos la matrícula detectada
                    matriculas_detectadas.append({
                        'bbox': (x1, y1, x2, y2),
                        'conf': conf_plate_det,
                        'texto': texto_matricula
                    })
        
        # Asociamos la matrícula detectada con cada vehículo y actualizamos la mejor predicción
        for obj in objetos_trackeados:
            if obj['tipo'] in ['car', 'motorcycle', 'bus', 'truck']:
                x1v, y1v, x2v, y2v = obj['bbox']
                track_id = obj['track_id']
                for mat in matriculas_detectadas:
                    x1m, y1m, x2m, y2m = mat['bbox']
                    intersecta_x = max(0, min(x2v, x2m) - max(x1v, x1m))
                    intersecta_y = max(0, min(y2v, y2m) - max(y1v, y1m))
                    area_interseccion = intersecta_x * intersecta_y
                    if area_interseccion > 0 and mat['texto']:
                        # Actualizamos solo si es la primera detección o si tiene mayor confianza/longitud
                        if track_id not in mejores_matriculas_por_vehiculo or \
                           (mat['conf'] > mejores_matriculas_por_vehiculo[track_id]['conf'] or
                            len(mat['texto']) > len(mejores_matriculas_por_vehiculo[track_id]['texto'])):
                            mejores_matriculas_por_vehiculo[track_id] = {
                                'frame': frame_num,
                                'bbox': mat['bbox'],
                                'texto': mat['texto'],
                                'conf': mat['conf']
                            }
                        break  # solo una matrícula por vehículo

        # Dibujamos las matrículas detectadas en el frame (sin asignar ID)
        annotated = tracked_frame.copy()
        for mat in matriculas_detectadas:
            x1, y1, x2, y2 = mat['bbox']
            cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 0, 255), 2)
            if mat['texto']:
                texto = mat['texto']
                (text_width, text_height), _ = cv2.getTextSize(texto, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
                cv2.rectangle(annotated, (x1, y1-text_height-10), (x1+text_width+5, y1), (0, 0, 255), -1)
                cv2.putText(annotated, texto, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        
        # Guardamos los objetos del frame para el CSV
        objetos_por_frame.append((frame_num, objetos_trackeados))

        # Escribimos el frame al video
        writer.write(annotated)
    
    # Cerramos el video
    if writer is not None:
        writer.release()

print("\nGenerando CSV final filtrado por mejor matrícula por vehículo...")

# Escribimos CSV final respetando la asociación vehículo-matrícula
with open(csv_path, 'w', newline='', encoding='utf-8') as f_csv:
    csv_writer = csv.writer(f_csv)
    csv_writer.writerow([
        'frame', 'tipo', 'conf_obj', 'track_id', 'x1', 'y1', 'x2', 'y2',
        'mat_detectada', 'conf_mat', 'mat_x1', 'mat_y1', 'mat_x2', 'mat_y2', 'texto_matricula'
    ])

    for frame_num, objetos_trackeados in objetos_por_frame:
        for obj in objetos_trackeados:
            track_id = obj['track_id']
            # Si tenemos matrícula asociada al vehículo, usamos la mejor
            if track_id in mejores_matriculas_por_vehiculo:
                mat = mejores_matriculas_por_vehiculo[track_id]
                csv_writer.writerow([
                    mat['frame'],
                    obj['tipo'],
                    obj['conf'],
                    track_id,
                    *obj['bbox'],
                    'Si',
                    mat['conf'],
                    *mat['bbox'],
                    mat['texto']
                ])
            else:
                # Si no hay matrícula asociada
                csv_writer.writerow([
                    frame_num,
                    obj['tipo'],
                    obj['conf'],
                    track_id,
                    *obj['bbox'],
                    'No', '', '', '', '', ''
                ])

print(f"\nVideo guardado en: {output_path}")
print(f"CSV final filtrado guardado en: {csv_path}")
